# FisioClin — entrenamiento de la red temporalReconocimiento de 20 ejercicios terapéuticos de artrosis de cadera y rodillaa partir de secuencias de ángulos articulares.**Qué necesitas subir:** `ventanas.npz` y `clases.json`.**Qué produce:** métricas fuera de partición, matriz de confusión, curva deaprendizaje y el modelo entrenado.---### Tres reglas que este cuaderno respeta y conviene no tocar1. **La partición se agrupa por escena.** Las ventanas se solapan dos tercios:   dos consecutivas son casi el mismo dato. Repartirlas al azar da una   exactitud altísima y falsa.2. **El escalador se ajusta solo con el conjunto de entrenamiento.** Ajustarlo   sobre todo el dataset es fuga de información.3. **Se reporta media y desviación**, nunca una sola cifra, y siempre junto a   la línea base.

## 1. Preparación del entorno

In [ ]:
!pip install -q scikit-learnimport tensorflow as tf, sklearn, numpy as npprint("TensorFlow", tf.__version__, "| scikit-learn", sklearn.__version__)print("GPU:", tf.config.list_physical_devices("GPU") or "no disponible (CPU)")

## 2. Subir los datosEjecuta la celda y selecciona **`ventanas.npz`** y **`clases.json`**.Si prefieres tenerlos en Drive, comenta esta celda y usa la siguiente.

In [ ]:
from google.colab import filessubidos = files.upload()print("Archivos:", list(subidos))

In [ ]:
# Alternativa con Google Drive (descomenta si prefieres esta vía)# from google.colab import drive# drive.mount('/content/drive')# RUTA = '/content/drive/MyDrive/FisioClin/'RUTA = ''

## 3. Cargar y describir el conjunto de datos

In [ ]:
import json, numpy as npfrom collections import Counterd = np.load(RUTA + "ventanas.npz", allow_pickle=True)meta = json.load(open(RUTA + "clases.json", encoding="utf-8"))X       = d["X"].astype(np.float32)y_ej    = d["y_ejercicio"].astype(np.int32)y_ca    = d["y_calidad"].astype(np.int32)escena  = d["escena"]        # <-- la columna de agrupamientosegmento = d["segmento"]     # <-- la tanda, para el votoclases  = meta["ejercicios"]con_calidad = y_ca >= 0y_ca_seguro = np.where(con_calidad, y_ca, 0)cuenta = Counter(y_ej.tolist())base_ej = cuenta.most_common(1)[0][1] / len(y_ej)base_ca = (Counter(y_ca[con_calidad].tolist()).most_common(1)[0][1]           / int(con_calidad.sum()))print(f"X {X.shape}  ->  {X.shape[0]} ventanas de {X.shape[1]} muestras x {X.shape[2]} features")print(f"{len(clases)} clases | {len(set(escena.tolist()))} escenas | {len(set(segmento.tolist()))} tandas")print(f"ventanas por clase: min {min(cuenta.values())} max {max(cuenta.values())}"      f"  (desbalance {max(cuenta.values())/min(cuenta.values()):.1f}x)")print(f"con etiqueta de calidad: {int(con_calidad.sum())} de {len(y_ca)}")print()print(f"LINEA BASE ejercicio: {base_ej:.3f}   (clase mayoritaria)")print(f"LINEA BASE calidad:   {base_ca:.3f}")print()print("Features:", ", ".join(meta["features"]))

### Distribución por claseEl desbalance obliga a compensar con pesos por muestra **y** a reportar F1macro además de exactitud: la exactitud sola esconde las clases pequeñas.

In [ ]:
import matplotlib.pyplot as pltorden = sorted(range(len(clases)), key=lambda i: -cuenta.get(i, 0))plt.figure(figsize=(10, 5))plt.barh([clases[i][:34] for i in orden][::-1],         [cuenta.get(i, 0) for i in orden][::-1], color="#4472C4")plt.xlabel("ventanas"); plt.title("Ventanas disponibles por ejercicio")plt.tight_layout(); plt.show()

## 4. La red

In [ ]:
from tensorflow import kerasfrom tensorflow.keras import layersEPOCAS, PACIENCIA, LOTE, APRENDIZAJE = 80, 15, 32, 1e-3def construir(n_pasos, n_features, n_clases, n_calidad=3):    entrada = keras.Input(shape=(n_pasos, n_features), name="secuencia")    x = layers.LSTM(64, return_sequences=True)(entrada)    x = layers.Dropout(0.3)(x)    x = layers.LSTM(32)(x)    x = layers.Dropout(0.3)(x)    tronco = layers.Dense(32, activation="relu")(x)    # Dos cabezas sobre el mismo tronco: reconocer el ejercicio y juzgar    # su ejecucion comparten la representacion del movimiento.    salida_ej = layers.Dense(n_clases, activation="softmax", name="ejercicio")(tronco)    salida_ca = layers.Dense(n_calidad, activation="softmax", name="calidad")(tronco)    modelo = keras.Model(entrada, [salida_ej, salida_ca])    modelo.compile(        optimizer=keras.optimizers.Adam(APRENDIZAJE),        loss=["sparse_categorical_crossentropy"] * 2,        loss_weights=[1.0, 0.3],   # la cabeza de calidad no debe dominar    )    return modeloconstruir(X.shape[1], X.shape[2], len(clases)).summary()

## 5. Utilidades: pesos, escalado y voto por tanda

In [ ]:
from collections import defaultdictdef pesos_por_clase(y, mascara=None):    """Compensa el desbalance de clases."""    w = np.zeros(len(y), dtype=np.float32)    valido = np.ones(len(y), bool) if mascara is None else mascara    c = Counter(y[valido].tolist()); n = int(valido.sum())    for i in np.where(valido)[0]:        w[i] = n / (len(c) * c[y[i]])    return wdef escalar(train, test):    """Media y desviacion del ENTRENAMIENTO, aplicadas a ambos."""    plano = train.reshape(-1, train.shape[-1])    media, desv = plano.mean(axis=0), plano.std(axis=0)    desv[desv < 1e-6] = 1.0    return (train - media) / desv, (test - media) / desv, media, desvdef voto_por_tanda(segmentos, y_real, y_pred):    """Exactitud por tanda: la unidad que ve el paciente."""    votos, reales = defaultdict(list), {}    for s, r, p in zip(segmentos, y_real, y_pred):        votos[s].append(p); reales[s] = r    ok = sum(1 for s, v in votos.items() if Counter(v).most_common(1)[0][0] == reales[s])    return ok / len(votos), len(votos)

## 6. Validación cruzada agrupada`StratifiedGroupKFold` mantiene las escenas enteras de un lado o del otro **y**reparte las clases entre particiones.Agrupar por *video* sería más exigente y es imposible: cada clase está en unsolo video, así que apartar un video deja a esa clase sin ejemplos. Por eso lacifra mide **generalización a otro plano de cámara, no a otro cuerpo**.

In [ ]:
from sklearn.model_selection import StratifiedGroupKFoldfrom sklearn.metrics import f1_score, confusion_matriximport pandas as pdFOLDS, SEMILLAS = 5, 3filas = []matriz = np.zeros((len(clases), len(clases)), dtype=int)for semilla in range(SEMILLAS):    keras.utils.set_random_seed(1000 + semilla)    gkf = StratifiedGroupKFold(n_splits=FOLDS, shuffle=True, random_state=1000 + semilla)    for fold, (tr, te) in enumerate(gkf.split(X, y_ej, groups=escena), 1):        Xtr, Xte, _, _ = escalar(X[tr], X[te])        modelo = construir(X.shape[1], X.shape[2], len(clases))        parada = keras.callbacks.EarlyStopping(monitor="val_loss",                                               patience=PACIENCIA,                                               restore_best_weights=True)        h = modelo.fit(Xtr, [y_ej[tr], y_ca_seguro[tr]],                       sample_weight=[pesos_por_clase(y_ej[tr]),                                      pesos_por_clase(y_ca_seguro[tr], con_calidad[tr])],                       validation_data=(Xte, [y_ej[te], y_ca_seguro[te]]),                       epochs=EPOCAS, batch_size=LOTE, verbose=0, callbacks=[parada])        p_ej, p_ca = modelo.predict(Xte, verbose=0)        pred = p_ej.argmax(axis=1)        m = con_calidad[te]        tanda, n_t = voto_por_tanda(segmento[te], y_ej[te], pred)        matriz += confusion_matrix(y_ej[te], pred, labels=list(range(len(clases))))        filas.append({            "semilla": semilla, "fold": fold, "epocas": len(h.history["loss"]),            "exactitud_ventana": float((pred == y_ej[te]).mean()),            "f1_macro": float(f1_score(y_ej[te], pred, average="macro", zero_division=0)),            "exactitud_tanda": float(tanda),            "exactitud_calidad": float((p_ca.argmax(axis=1)[m] == y_ca[te][m]).mean())                                  if m.any() else np.nan,        })        print(f"  semilla {semilla} fold {fold}: ventana {filas[-1]['exactitud_ventana']:.3f}"              f"  f1 {filas[-1]['f1_macro']:.3f}  tanda {tanda:.3f}"              f"  [{filas[-1]['epocas']} epocas]", flush=True)res = pd.DataFrame(filas)res.to_csv("metricas_validacion.csv", index=False)print("\nCorridas:", len(res))

## 7. Resultados

In [ ]:
print(f"RESULTADO FUERA DE PARTICION ({len(res)} corridas)\n")for col, nombre in [("exactitud_ventana", "Ejercicio por ventana de 3 s"),                    ("f1_macro",          "Ejercicio F1 macro"),                    ("exactitud_tanda",   "Ejercicio por tanda (voto)"),                    ("exactitud_calidad", "Calidad (cabeza de la red)")]:    print(f"  {nombre:32} {res[col].mean():.3f} +- {res[col].std():.3f}")print(f"\n  {'LINEA BASE ejercicio':32} {base_ej:.3f}")print(f"  {'LINEA BASE calidad':32} {base_ca:.3f}")print("\nRango entre particiones: "      f"{res['exactitud_ventana'].min():.3f} a {res['exactitud_ventana'].max():.3f}")print("Por eso se reporta media y desviacion, nunca la mejor particion.")

**Si la cabeza de calidad queda por debajo de su línea base, no es un fallo.**Aprende a reproducir reglas que ya se calculan exacto desde los mismos ángulos,así que sólo puede perder información. Es la demostración de que la evaluaciónde la ejecución le corresponde a las reglas biomecánicas, no a la red.

## 8. Matriz de confusión

In [ ]:
import matplotlib.pyplot as pltnorm = matriz / np.maximum(matriz.sum(axis=1, keepdims=True), 1)fig, ax = plt.subplots(figsize=(11, 9))im = ax.imshow(norm, cmap="Blues", vmin=0, vmax=1)ax.set_xticks(range(len(clases))); ax.set_yticks(range(len(clases)))ax.set_xticklabels([c[:26] for c in clases], rotation=90, fontsize=7)ax.set_yticklabels([c[:26] for c in clases], fontsize=7)ax.set_xlabel("predicho"); ax.set_ylabel("real")ax.set_title("Matriz de confusion (fuera de particion, acumulada)")fig.colorbar(im, fraction=0.046)plt.tight_layout(); plt.savefig("matriz_confusion.png", dpi=130); plt.show()rec = matriz.diagonal() / np.maximum(matriz.sum(axis=1), 1)prec = matriz.diagonal() / np.maximum(matriz.sum(axis=0), 1)f1 = 2 * rec * prec / np.maximum(rec + prec, 1e-9)tabla = pd.DataFrame({"ejercicio": clases, "recall": rec.round(2),                      "precision": prec.round(2), "f1": f1.round(2),                      "n": matriz.sum(axis=1)}).sort_values("f1", ascending=False)tabla.to_csv("metricas_por_clase.csv", index=False)tabla

## 9. Curva de aprendizajeResponde a la pregunta que decide si vale la pena capturar más material: **¿lapendiente ya se agotó?** Tarda varios minutos.

In [ ]:
FRACCIONES = [0.25, 0.50, 0.75, 1.00]curva = []gkf = StratifiedGroupKFold(n_splits=FOLDS, shuffle=True, random_state=1000)for fold, (tr, te) in enumerate(gkf.split(X, y_ej, groups=escena), 1):    escenas_tr = sorted(set(escena[tr].tolist()))    rng = np.random.default_rng(1000 + fold)    orden = rng.permutation(escenas_tr)    for frac in FRACCIONES:        n = max(2, int(round(frac * len(orden))))        elegidas = set(orden[:n].tolist())        sub = np.array([i for i in tr if escena[i] in elegidas])        keras.utils.set_random_seed(1000 + fold)        Xtr, Xte, _, _ = escalar(X[sub], X[te])        modelo = construir(X.shape[1], X.shape[2], len(clases))        parada = keras.callbacks.EarlyStopping(monitor="val_loss", patience=PACIENCIA,                                               restore_best_weights=True)        ceros_tr = np.zeros(len(sub), dtype=np.int32)        modelo.fit(Xtr, [y_ej[sub], ceros_tr],                   sample_weight=[pesos_por_clase(y_ej[sub]), ceros_tr * 0.0],                   validation_data=(Xte, [y_ej[te], np.zeros(len(te), dtype=np.int32)]),                   epochs=EPOCAS, batch_size=LOTE, verbose=0, callbacks=[parada])        pred = modelo.predict(Xte, verbose=0)[0].argmax(axis=1)        curva.append({"fold": fold, "fraccion": frac, "ventanas": len(sub),                      "exactitud": float((pred == y_ej[te]).mean())})        print(f"  fold {fold} {frac:.0%}: {len(sub)} ventanas -> {curva[-1]['exactitud']:.3f}",              flush=True)cur = pd.DataFrame(curva); cur.to_csv("curva_aprendizaje.csv", index=False)g = cur.groupby("fraccion").agg(ventanas=("ventanas", "mean"),                                media=("exactitud", "mean"),                                desv=("exactitud", "std")).reset_index()plt.figure(figsize=(8, 4.5))plt.errorbar(g["ventanas"], g["media"], yerr=g["desv"], marker="o", capsize=4)plt.axhline(base_ej, color="crimson", ls=":", label="linea base")plt.xlabel("ventanas de entrenamiento"); plt.ylabel("exactitud fuera de particion")plt.title("Curva de aprendizaje"); plt.legend(); plt.grid(alpha=.3)plt.tight_layout(); plt.savefig("curva_aprendizaje.png", dpi=130); plt.show()g

## 10. Modelo final y descarga**Aviso:** este modelo entrena con todas las escenas, así que su exactitudsobre este mismo material está contaminada. Sirve para desplegar, no parareportar. La cifra publicable es la de fuera de partición.

In [ ]:
keras.utils.set_random_seed(1000)plano = X.reshape(-1, X.shape[-1])media, desv = plano.mean(axis=0), plano.std(axis=0)desv[desv < 1e-6] = 1.0modelo = construir(X.shape[1], X.shape[2], len(clases))modelo.fit((X - media) / desv, [y_ej, y_ca_seguro],           sample_weight=[pesos_por_clase(y_ej),                          pesos_por_clase(y_ca_seguro, con_calidad)],           epochs=int(res["epocas"].mean()), batch_size=LOTE, verbose=0)modelo.save("modelo.keras")np.savez("escalador.npz", media=media, desviacion=desv)json.dump({**meta,           "validacion": {c: {"media": round(float(res[c].mean()), 4),                              "desviacion": round(float(res[c].std()), 4)}                          for c in ["exactitud_ventana", "f1_macro",                                    "exactitud_tanda", "exactitud_calidad"]},           "linea_base_ejercicio": round(base_ej, 4),           "linea_base_calidad": round(base_ca, 4),           "particion": "StratifiedGroupKFold por escena",           "aviso": "El modelo final entreno con todas las escenas: su exactitud "                    "sobre este material esta contaminada."},          open("modelo.json", "w", encoding="utf-8"), indent=2, ensure_ascii=False)print("Guardado: modelo.keras, escalador.npz, modelo.json")

In [ ]:
from google.colab import filesfor f in ["modelo.keras", "escalador.npz", "modelo.json",          "metricas_validacion.csv", "metricas_por_clase.csv",          "matriz_confusion.png", "curva_aprendizaje.png"]:    try:        files.download(f)    except Exception as e:        print("no se pudo descargar", f, e)